# Twitch TLG — degree params (σ, α) and KL per model per network

Consults the `scripts/closedform/run_tlg_twitch_gic.py` experiment outputs
(`runs/tlg_twitch_gic/<region>_table.csv`) for the **KL per family per network** (the six
Twitch country graphs), and reports the **degree-feature parameters σ and α** for each
network.

- **KL** comes straight from the experiment's per-region tables.
- **σ, α** are the intercept and degree slope of the degree-only logistic MLE
  `logit P[edge] = σ + α·D`, `D_ij = log(1+S_i)+log(1+S_j)` (d=1) — the same fit the
  experiment uses to get α (it only persists α, not σ, so we recompute both here).

In [ ]:
import sys, glob
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx

def find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "scripts" / "closedform" / "runs" / "tlg_twitch_gic").exists():
            return base
    raise RuntimeError("tlg_twitch_gic runs not found")
ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
from logit_graph.lg_features import build_pair_dataset
from logit_graph.temporal import fit_growth_params

TWITCH_RUNS = ROOT / "scripts" / "closedform" / "runs" / "tlg_twitch_gic"
DATA = ROOT / "data" / "twitch" / "graphs_processed"
DEG_D = 1
print("twitch runs:", TWITCH_RUNS)

### KL per model per network (consulted from the experiment tables)

In [ ]:
# read each region table -> KL per (region, model)
rows = []
for f in sorted(glob.glob(str(TWITCH_RUNS / "*_table.csv"))):
    t = pd.read_csv(f)
    t = t[t.model != "Real"]
    for _, r in t.iterrows():
        rows.append(dict(region=r["region"], model=r["model"], kl=r["kl"]))
kl_long = pd.DataFrame(rows)
kl_wide = kl_long.pivot_table(index="region", columns="model", values="kl")
MODELS = ["TLG", "SBM", "ER", "BA", "WS", "KR", "GRG"]
kl_wide = kl_wide[[m for m in MODELS if m in kl_wide.columns]]
kl_wide.round(4)

### Degree-feature params σ and α per network (degree-only logistic MLE)

In [ ]:
def load_region(path):
    G = nx.read_edgelist(path, comments="#", nodetype=int)
    G = nx.Graph(G); G.remove_edges_from(nx.selfloop_edges(G))
    cc = max(nx.connected_components(G), key=len)
    return nx.convert_node_labels_to_integers(G.subgraph(cc).copy())

deg_rows = []
for region in kl_wide.index:
    path = DATA / f"{region}_graph.edges"
    if not path.exists():
        print(f"  {region}: data file missing -> sigma/alpha NaN"); 
        deg_rows.append(dict(region=region, n=np.nan, sigma=np.nan, alpha=np.nan)); continue
    G = load_region(path)
    adj = nx.to_numpy_array(G)
    D, lab = build_pair_dataset(adj, d=DEG_D, mode="bounded", layer2=True)
    fit = fit_growth_params(D, lab)   # sigma, alpha (+ SEs) of logit P = sigma + alpha*D
    deg_rows.append(dict(region=region, n=G.number_of_nodes(),
                         sigma=fit["sigma"], alpha=fit["alpha"],
                         se_sigma=fit["se_sigma"], se_alpha=fit["se_alpha"]))
    print(f"  {region}: n={G.number_of_nodes()}  σ={fit['sigma']:+.3f}  α={fit['alpha']:+.3f}")
deg = pd.DataFrame(deg_rows).set_index("region")
deg.round(3)

### Combined table: degree params (σ, α) + KL per model, per network

One row per Twitch network: the degree-feature parameters and the KL of every family.

In [ ]:
table = deg[["n", "sigma", "alpha"]].join(kl_wide)
table = table.sort_values("n")

def _hl_min_kl(row):
    kl = row[kl_wide.columns]
    best = kl.idxmin()
    return ["font-weight: bold; color: #0072B2" if c == best else "" for c in row.index]

styled = (table.style
          .format({"sigma": "{:+.3f}", "alpha": "{:+.3f}", "n": "{:.0f}",
                   **{m: "{:.4f}" for m in kl_wide.columns}})
          .apply(_hl_min_kl, axis=1)
          .set_caption("Twitch: σ, α (degree MLE) and KL per model (bold = lowest KL / best fit)"))
display(styled)
table.to_csv("twitch_params_kl_table.csv")
print("saved twitch_params_kl_table.csv")
table.round(4)